# Anveshak: Open-World Remote Sensing Object Detector
### Techfest 2026-27 | IITB-HRTI Centre of Excellence
---
This turnkey notebook handles GPU training, continual few-shot learning, and evaluation directly inside Google Colab with seamless Google Drive & GitHub synchronization.

## 1. Verify GPU & Environment Setup

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

## 2. Mount Google Drive
Your persistent checkpoints, datasets, and submissions are stored in Google Drive.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_ANVESHAK_DIR = '/content/drive/MyDrive/Anveshak'
os.makedirs(f"{DRIVE_ANVESHAK_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE_ANVESHAK_DIR}/submissions", exist_ok=True)
print("[✓] Google Drive connected successfully.")

## 3. Setup Project Codebase
Choose **either** GitHub clone or Google Drive folder:

In [ ]:
# If using GitHub:
GITHUB_REPO_URL = ""  # Paste your repo URL here, e.g., https://github.com/username/IITBBB.git

if GITHUB_REPO_URL.strip():
    %cd /content
    if not os.path.exists("IITBBB"):
        !git clone {GITHUB_REPO_URL} IITBBB
    else:
        %cd IITBBB
        !git pull
    %cd /content/IITBBB
elif os.path.exists(f"{DRIVE_ANVESHAK_DIR}/IITBBB"):
    print("Loading project from Google Drive...")
    %cd {DRIVE_ANVESHAK_DIR}/IITBBB
else:
    # Working directly in Colab local directory
    %cd /content
    !mkdir -p IITBBB
    %cd /content/IITBBB

# Install dependencies
!pip install -r requirements.txt -q
!pip install gdown -q

## 4. Download & Extract Round 1 Dataset (20 GB) Directly to Colab SSD
Using Google's internal cloud connection, `round1.zip` downloads in ~1-2 minutes straight into fast local NVMe SSD (`/content/data`).

In [ ]:
import os
import zipfile

!mkdir -p /content/data
DATASET_FILE_ID = "1A24PTWWji509iDwf50EkJuTcwyb13RDH"
ZIP_PATH = "/content/data/round1.zip"

# 1. Download round1.zip directly at 100-200 MB/s
if not os.path.exists(ZIP_PATH):
    print(f"[*] Downloading round1.zip (20 GB) from Google Drive...")
    !gdown {DATASET_FILE_ID} -O {ZIP_PATH}

# 2. Extract directly into fast local SSD
print("[*] Extracting round1.zip into /content/data/...")
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall("/content/data/")

print("[✓] Round 1 Dataset unpacked successfully:")
!ls -lh /content/data

## 5. Train Detector on Set A (From Scratch - Random Initialization)

In [ ]:
# Execute Set A training script
!python -m src.training.train_set_a \
    --config configs/base_config.yaml \
    --data_dir /content/data \
    --output_dir checkpoints/

# Back up Set A weights to Google Drive immediately
!cp checkpoints/set_a_best.pth {DRIVE_ANVESHAK_DIR}/checkpoints/

## 6. Continual Few-Shot Learning on Set B
Continues the *exact same model lineage* on Set B (10 new classes) with distillation loss and cosine classifier margin to retain Set A classes.

In [ ]:
!python -m src.training.train_set_b \
    --config configs/base_config.yaml \
    --checkpoint_in checkpoints/set_a_best.pth \
    --data_dir /content/data \
    --output_checkpoint checkpoints/round1_final.pth

# Sync final checkpoint to Google Drive
!cp checkpoints/round1_final.pth {DRIVE_ANVESHAK_DIR}/checkpoints/
print("[✓] Round 1 Checkpoint saved and backed up to Google Drive.")

## 7. Inference & Official Conflict Rule Resolution
Generates raw predictions on unlabelled Round 1 test images, applies conflict filtering (IoU >= 0.5 highest score rule), and exports `predictions.csv`.

In [ ]:
!python -m src.inference.predict \
    --checkpoint checkpoints/round1_final.pth \
    --test_images_dir /content/data/test_images \
    --output_csv submission/predictions.csv

# Copy predictions to Google Drive
!cp submission/predictions.csv {DRIVE_ANVESHAK_DIR}/submissions/

## 8. Validate Predictions with Organizer's Script

In [ ]:
# Run validate_predictions.py
!python validate_predictions.py submission/predictions.csv